# Imports, Helpers, Parameters

### Imports

In [45]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from textwrap import wrap
import numpy as np
import os
import re
import matplotlib.font_manager as fm




### Parameters

In [46]:
main_color = "#3a5f83"

font_dir = r"C:\Users\teddy\Downloads\OAIPR\Technical\AEI Data Other\Lato"

# Loop through every file in the folder
for font_file in os.listdir(font_dir):
    if font_file.lower().endswith(".ttf") and "lato" in font_file.lower():
        font_path = os.path.join(font_dir, font_file)
        fm.fontManager.addfont(font_path)

plt.rcParams["font.family"] = "Lato"
plt.rcParams["font.weight"] = "normal"

palette = sns.color_palette("colorblind")
# Put once at the TOP of your notebook/script (or just tweak this line)
sns.set_context("notebook", font_scale=1.0)  # was 1.2; smaller = less crowded

chart_size = (17, 9)

outdir = "../outputs/charts_for_sharing/prelim_report_oct_2025/comparisons/v2_above_auto_median"
os.makedirs(outdir, exist_ok=True)

# Automation By AI Task Coverage

## Load Data

In [47]:
automation_tasks_imputed = pd.read_csv("../data/automation_tasks_imputed_v2_above_auto_median.csv")


## Tasks Automated

### % Major Occupational Cateogory Automated

In [48]:
# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level - both national and Utah
grouped = subset.groupby("major_occ_category").agg({
    "ai_task_comp_nat": "sum",
    "task_comp_nat": "sum",
    "ai_task_comp_ut": "sum",
    "task_comp_ut": "sum"
}).reset_index()

# Compute automation percentages for both
grouped["pct_automated_nat"] = (grouped["ai_task_comp_nat"] / grouped["task_comp_nat"]) * 100
grouped["pct_automated_ut"] = (grouped["ai_task_comp_ut"] / grouped["task_comp_ut"]) * 100

# Sort by NATIONAL percentage (descending)
grouped = grouped.sort_values("pct_automated_nat", ascending=False).reset_index(drop=True)

# Reshape for grouped bar chart
grouped_melted = grouped.melt(
    id_vars=['major_occ_category'],
    value_vars=['pct_automated_nat', 'pct_automated_ut'],
    var_name='region',
    value_name='pct_automated'
)

# Create figure with larger width for grouped bars
fig, ax = plt.subplots(figsize=chart_size)

# Define colors
colors = {'pct_automated_nat': '#22688F', 'pct_automated_ut': '#759b8f'}

# Create grouped bar plot
sns.barplot(data=grouped_melted, x='pct_automated', y='major_occ_category', 
            hue='region', palette=colors, ax=ax)

# Add value labels at end of bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=8)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Major Occupational Categories Automated by AI Task Coverage: National vs. Utah", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Percent of Category Tasks Automated (%)", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Update legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, ['National', 'Utah'], loc='lower right', fontsize=10, title='Region')

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Percentages are identical for National and Utah (same task frequencies). Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "% Major Occupational Category Automated.png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Average % Occupation Automated By Major Occupational Category

In [49]:
# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "people_automated_nat": "sum",
    "ai_task_comp_nat": "sum",
    "task_comp_nat": "sum",
    "freq_sum_eco": "sum",
    "freq_sum_ai": "sum",
    "pct_automated_nat": "mean"  
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_nat_moc"] = (grouped["ai_task_comp_nat"] / grouped["task_comp_nat"]) * 100

# CHART ALICE ASKED FOR
# Chart for Ttile and filename: "% Task Frequencies Automated By Major Occupational Category"
# X axis label: % Task Frequencies Automated (% Maj Occ Cat Tasks Automated)
# Replace the pct_automated_nat with pct_automated_nat_freq for this new chart
# grouped["pct_automated_nat_freq"] = (grouped["freq_sum_ai"] / grouped["freq_sum_eco"]) * 100

# Sort by automation percentage and get all categories
grouped = grouped.sort_values("pct_automated_nat", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='pct_automated_nat', y="major_occ_category", 
            color=main_color, ax=ax)

# Add value labels at end of bars
for i, (idx, row) in enumerate(grouped.iterrows()):
    value = row['pct_automated_nat']
    pct = row['pct_automated_nat_moc']
    ax.text(value, i, f' {value:.1f}% ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)

# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.05)  # Add 5% padding on right side

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Average % Occupation Automated By Major Occupational Category By AI Task Coverage", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Avg % Occ Automated (% Maj Occ Cat Automated)", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Assumes all tasks take the same amount of time. Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Average % Occupation Automated In Each Major Occupational Category.png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Top 15 % Occupation Automated

In [50]:
# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Get top 15 most automated occupations
top_15 = subset.nlargest(15, 'pct_automated_nat').copy()

# Create combined label with title and major category
top_15['title_with_category'] = top_15.apply(
    lambda row: f"{row['title']} [{row['major_occ_category']}]",
    axis=1
)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use a professional color palette
palette = sns.color_palette("colorblind")
sns.barplot(data=top_15, x='pct_automated_nat', y='title_with_category', 
            color=main_color, ax=ax)

# Add value labels at end of bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=65)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.42)

# Professional titles and labels
ax.set_title("Top 15 Most Automated Occupations by AI Task Coverage", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Percent of Occupation Tasks Automated (%)", fontsize=12, labelpad=10)
ax.set_ylabel("Occupation [Major Occupational Category]", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Top 15 % Occupation Automated.png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

## Workers Automated

### Workers Automated by Major Occupational Category National

In [51]:
# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "people_automated_nat": "sum",
    "ai_task_comp_nat": "sum",
    "task_comp_nat": "sum"
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_nat"] = (grouped["ai_task_comp_nat"] / grouped["task_comp_nat"]) * 100

# Sort by automation percentage and get all categories (there aren't that many)
grouped = grouped.sort_values("people_automated_nat", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='people_automated_nat', y="major_occ_category", 
            color=main_color, ax=ax)

# Add value labels at end of bars
for i, (idx, row) in enumerate(grouped.iterrows()):
    value = row['people_automated_nat']
    pct = row['pct_automated_nat']
    ax.text(value, i, f' {value:,.0f} ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)

# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.05)  # Add 5% padding on right side

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Workers Automated by Major Occupational Category by AI Task Coverage National", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Automated (% Maj Occ Cat Automated)", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Assumes all tasks take the same amount of time. Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Workers Automated by Major Occupational Category National.png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Workers Automated by Major Occupational Category Utah

In [52]:
# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "people_automated_ut": "sum",
    "ai_task_comp_ut": "sum",
    "task_comp_ut": "sum"
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_ut"] = (grouped["ai_task_comp_ut"] / grouped["task_comp_ut"]) * 100

# Sort by automation percentage and get all categories (there aren't that many)
grouped = grouped.sort_values("people_automated_ut", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='people_automated_ut', y="major_occ_category", 
            color=main_color, ax=ax)

# Add value labels at end of bars
for i, (idx, row) in enumerate(grouped.iterrows()):
    value = row['people_automated_ut']
    pct = row['pct_automated_ut']
    ax.text(value, i, f' {value:,.0f} ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)

# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.02)  # Add 2% padding on right side

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Workers Automated by Major Occupational Category by AI Task Coverage Utah", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Automated Utah (% Maj Occ Cat Automated)", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Assumes all tasks take the same amount of time. Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Workers Automated by Major Occupational Category Utah.png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Top 15 Workers Automated by Occupation National

In [53]:
# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Get top 15 most automated occupations
top_15 = subset.nlargest(15, 'people_automated_nat').copy()

# Create combined label with title and major category
top_15['title_with_category'] = top_15.apply(
    lambda row: f"{row['title']} [{row['major_occ_category']}]",
    axis=1
)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use a professional color palette
palette = sns.color_palette("colorblind")
sns.barplot(data=top_15, x='people_automated_nat', y='title_with_category', 
            color=main_color, ax=ax)

# Add custom labels showing both count and percentage
for i, (idx, row) in enumerate(top_15.iterrows()):
    value = row['people_automated_nat']
    pct = row['pct_automated_nat']
    ax.text(value, i, f' {value:,.0f} ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)

# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.05)  # More padding for longer labels

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=65)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.42)

# Professional titles and labels
ax.set_title("Top 15 Number of Workers Automated by Occupation by AI Task Coverage National", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Automated National (% Occ Automated)", fontsize=12, labelpad=10)
ax.set_ylabel("Occupation [Major Occupational Category]", fontsize=12, labelpad=10)

# Format x-axis
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Assumes all tasks take the same amount of time. Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Top 15 Workers Automated by Occupation National.png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Top 15 Workers Automated by Occupation Utah

In [54]:
# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Get top 15 most automated occupations
top_15 = subset.nlargest(15, 'people_automated_ut').copy()

# Create combined label with title and major category
top_15['title_with_category'] = top_15.apply(
    lambda row: f"{row['title']} [{row['major_occ_category']}]",
    axis=1
)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use a professional color palette
palette = sns.color_palette("colorblind")
sns.barplot(data=top_15, x='people_automated_ut', y='title_with_category', 
            color=main_color, ax=ax)

for i, (idx, row) in enumerate(top_15.iterrows()):
    value = row['people_automated_ut']
    pct = row['pct_automated_nat']
    ax.text(value, i, f' {value:,.0f} ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)

# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.05)  # Add 5% padding on right side

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=65)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.42)

# Professional titles and labels
ax.set_title("Top 15 Number of Workers Automated by Occupation by AI Task Coverage Utah ", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Automated Utah (% Occ Automated)", fontsize=12, labelpad=10)
ax.set_ylabel("Occupation [Major Occupational Category]", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Assumes all tasks take the same amount of time. Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Top 15 Workers Automated by Occupation Utah.png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

## Economic Value of Automation

### Economic Value Generated by Major Occupational Category National

In [55]:
# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "eco_value_nat": "sum",
    "ai_task_comp_nat": "sum",
    "task_comp_nat": "sum"
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_nat"] = (grouped["ai_task_comp_nat"] / grouped["task_comp_nat"]) * 100

# Sort by economic value and get all categories
grouped = grouped.sort_values("eco_value_nat", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='eco_value_nat', y="major_occ_category", 
            color=main_color, ax=ax)

# Add labels with dollar amount and percentage
for i, (idx, row) in enumerate(grouped.iterrows()):
    value = row['eco_value_nat']
    pct = row['pct_automated_nat']
    ax.text(value, i, f' ${value:,.0f} ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)
    
# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.1)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Economic Value Generated by Major Occupational Category National", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Economic Value Generated in $$$ (% Maj Occ Cat Automated)", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as currency
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Economic Value by Major Occupational Category National.png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Economic Value Generated by Major Occupational Category Utah

In [56]:
# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "eco_value_ut": "sum",
    "ai_task_comp_ut": "sum",
    "task_comp_ut": "sum"
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_ut"] = (grouped["ai_task_comp_ut"] / grouped["task_comp_ut"]) * 100

# Sort by economic value and get all categories
grouped = grouped.sort_values("eco_value_ut", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='eco_value_ut', y="major_occ_category", 
            color=main_color, ax=ax)

# Add labels with dollar amount and percentage
for i, (idx, row) in enumerate(grouped.iterrows()):
    value = row['eco_value_ut']
    pct = row['pct_automated_ut']
    ax.text(value, i, f' ${value:,.0f} ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)
    
# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.1)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Economic Value Generated by Major Occupational Category Utah", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Economic Value Generated in $$$ (% Maj Occ Cat Automated)", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as currency
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Economic Value by Major Occupational Category Utah.png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Top 15 Most Economic Value Generated by Occupation National

In [57]:
# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Get top 15 most automated occupations
top_15 = subset.nlargest(15, 'eco_value_nat').copy()

# Create combined label with title and major category
top_15['title_with_category'] = top_15.apply(
    lambda row: f"{row['title']} [{row['major_occ_category']}]",
    axis=1
)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use a professional color palette
palette = sns.color_palette("colorblind")
sns.barplot(data=top_15, x='eco_value_nat', y='title_with_category', 
            color=main_color, ax=ax)


for i, (idx, row) in enumerate(top_15.iterrows()):
    value = row['eco_value_nat']
    pct = row['pct_automated_nat']
    ax.text(value, i, f' ${value:,.0f} ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)
    
# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.1)  # More padding for longer labels

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=65)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.42)

# Professional titles and labels
ax.set_title("Top 15 Most Economic Value Generated by Occupation by AI Task Coverage National", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Economic Value Generated in $$$ (% Occ Automated)", fontsize=12, labelpad=10)
ax.set_ylabel("Occupation [Major Occupational Category]", fontsize=12, labelpad=10)

# Format x-axis as currency
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Top 15 Most Economic Value Generated by Occupation National.png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Top 15 Most Economic Value Generated by Occupation Utah

In [58]:
# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Get top 15 most automated occupations
top_15 = subset.nlargest(15, 'eco_value_ut').copy()

# Create combined label with title and major category
top_15['title_with_category'] = top_15.apply(
    lambda row: f"{row['title']} [{row['major_occ_category']}]",
    axis=1
)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use a professional color palette
palette = sns.color_palette("colorblind")
sns.barplot(data=top_15, x='eco_value_ut', y='title_with_category', 
            color=main_color, ax=ax)


for i, (idx, row) in enumerate(top_15.iterrows()):
    value = row['eco_value_ut']
    pct = row['pct_automated_nat']
    ax.text(value, i, f' ${value:,.0f} ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)
    
# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.1)  # More padding for longer labels

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=65)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.42)

# Professional titles and labels
ax.set_title("Top 15 Most Economic Value Generated by Occupation by AI Task Coverage Utah", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Economic Value Generated in $$$ (% Occ Automated)", fontsize=12, labelpad=10)
ax.set_ylabel("Occupation [Major Occupational Category]", fontsize=12, labelpad=10)

# Format x-axis as currency
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Top 15 Most Economic Value Generated by Occupation Utah.png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)